# انسان در حلقه: دروازه‌های پیش از اقدام، طبقه‌بندی ریسک، و ثبت گزارش حسابرسی

فایل README این درس انسان در حلقه را با قطعه‌کدی کوتاه معرفی می‌کند که پس از آنکه عامل پاسخ تولید کرد، از کاربر می‌پرسد `تایید` یا `رد` کند. آن الگو نقطه شروع خوبی است، اما پیاده‌سازی‌های تولیدی HITL معمولاً به سه بخش اضافی نیاز دارند:

1. یک **دروازه پیش از اقدام** که **قبل از** اجرای گام پرریسک توسط عامل، اجرا شود، تا هزینه، غیرقابل بازگشت بودن، و تاخیر تحت کنترل باقی بمانند.
2. **طبقه‌بندی ریسک**، تا اقدامات کم‌ریسک به‌طور خودکار اجرا شوند، اقدامات با ریسک متوسط به صورت دسته‌ای تأیید گردند، و فقط اقدامات پرریسک بر روی انسان متوقف شوند.
3. یک **ثبت گزارش حسابرسی به همراه حلقه بازنگری**، به‌طوری که هر تصمیم دروازه به صورت JSONL ثبت شود، و رد شدن بازپرسش عامل با دلیل ساختاریافته انجام شود به جای اینکه فقط `در حال بازنگری...` چاپ شود.

این دفترچه یادداشت هرکدام از این‌ها را بر پایه همان اصول ساده‌ای که در `06-system-message-framework.ipynb` وجود دارد، می‌سازد. این برنامه از ابتدا تا انتها در حالت `DEMO_MODE = True` (بدون نیاز به ورودی تعاملی) اجرا می‌شود یا با درخواست‌های ورودی واقعی `input()` وقتی `DEMO_MODE = False`. توجه: در حالت DEMO عمل تلاش مجدد هدف سوم به صورت اسکریپتی اجرا می‌شود تا مکانیزم حلقه از ابتدا تا انتها قابل مشاهده باشد. بازطبقه‌بندی مبتنی بر بازبینی واقعی نیازمند `DEMO_MODE = False` و یک اپراتور است.

**خارج از حوزه (در درس‌های دیگر بررسی می‌شود):** احراز هویت و کنترل دسترسی (تهدید ۲ در README درس ۰۶)، نرم‌افزار میانجی فراخوانی ابزار (بررسی عمیق MAF در درس ۱۴)، الگوهای مناظره چندعامله.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## الگو ۱: دروازه پیش از عمل

بخش HITL در README ابتدا عامل را فرا می‌خواند، سپس از کاربر می‌خواهد خروجی را تأیید کند. این یک جریان **پس از عمل** است. عامل قبلاً اجرا شده است، بنابراین هزینه تماس با LLM پرداخت شده و هر اثر جانبی (ایمیل ارسال شده، ردیف پایگاه داده نوشته شده، نظر ارسال شده) قبلاً اتفاق افتاده است.

یک جریان **پیش از عمل** دروازه را قبل از اجرای مرحله پرخطر توسط عامل قرار می‌دهد. عامل عمل را پیشنهاد می‌دهد، دروازه تصمیم می‌گیرد که آیا اجرا شود، و تنها با تأیید، اثر جانبی رخ می‌دهد.

| جنبه | تأیید پس از عمل (بخش README) | دروازه پیش از عمل (این دفترچه یادداشت) |
|---|---|---|
| تأیید کی انجام می‌شود؟ | بعد از اینکه عامل خروجی تولید کرده است | قبل از اجرای هر اثر جانبی |
| هزینه LLM در صورت رد | قبلاً پرداخت شده است | فقط برای پیشنهاد، نه عمل پرداخت شده است |
| اثرات جانبی غیرقابل برگشت | ممکن است (عمل قبلاً اتفاق افتاده است) | جلوگیری شده است |
| وضوح ممیزی | تأیید یک دستور چاپ است | تأیید یک رکورد JSON با زمان‌سنج، عمل، دلیل است |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## الگو ۳: گزارش حسابرسی و چرخه بازنگری  

یک `print("Response approved.")` گزارش حسابرسی نیست. برای اعتماد، هر تصمیم گیت باید به‌عنوان یک رخداد ساختاریافته ثبت شود که بعداً بتوانید آن را جستجو، بازپخش یا به مرور حادثه پیوست کنید.  

دو قسمت:  

۱. **فایل JSONL فقط الحاقی.** هر خط یک تصمیم با زمان‌بندی، اقدام، طبقه، تصمیم، و دلیل. آسان برای جستجو با grep، آسان برای ارسال به یک انبار گزارش واقعی بعداً.  
۲. **چرخه بازنگری در صورت رد.** وقتی گیت `deny` برمی‌گرداند، عامل با دلیل رد در زمینه خودش را دوباره فرا می‌خواند، تا پیشنهاد بعدی بتواند از مشکل جلوگیری کند.  


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## منابع اضافی

چندین پروژه عمومی دیگر انواع مختلفی از این الگوهای HITL را پیاده‌سازی می‌کنند. رویکردها را مقایسه کنید تا ببینید کدام برای پشته شما مناسب است:

- **LangChain** بسته‌های ابزار انسان در حلقه ([مستندات](https://python.langchain.com/docs/integrations/tools/human_tools)): بسته‌های ابزار قابل جایگزینی که اجرای برنامه را برای ورودی انسانی متوقف می‌کنند.
- **AutoGen** `UserProxyAgent` ([مستندات v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); در نسخه v0.4+ ساختار این مورد تغییر کرده است): از نقش نماینده خاصی برای نمایش انسان در مکالمات چندعاملی استفاده می‌کند.
- **Microsoft Agent Framework (MAF)** واسطه فراخوانی تابع ([مستندات](https://learn.microsoft.com/agent-framework/)): واسطه‌ای که حول هر فراخوانی ابزار/تابع اجرا می‌شود، مناسب برای منطق کنترل دسترسی و جریان‌های تأیید.

هر پروژه این سه زیرالگو را به روش متفاوتی مدیریت می‌کند: LangChain آنها را به عنوان ابزار بسته‌بندی می‌کند، AutoGen از نقش عامل استفاده می‌کند، و Microsoft Agent Framework از واسطه فراخوانی تابع بهره می‌برد. یک یا دو پیاده‌سازی را به صورت کامل مطالعه کنید قبل از اینکه طراحی خود را برای عامل خود انتخاب کنید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
